<a href="https://colab.research.google.com/github/meharalirajar060-codeee/Flyrank_Internship_ML_MAR/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/meharalirajar060-codeee/Flyrank_Internship_ML_MAR/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
"""One row = one content page (content_hash_id), for one client (client_hash_id), on one calendar day (report_date), from fact_content_daily_performance. For my Lane 2 decision, I roll many daily rows up into one row per content page per feature window.

Time window: a mid-panel month, month=2026-03, for feature-building — never the _sample table, which is the final month (June 2026) and would leak the future outcome window into label logic."""

'One row = one content page (content_hash_id), for one client (client_hash_id), on one calendar day (report_date), from fact_content_daily_performance. For my Lane 2 decision, I roll many daily rows up into one row per content page per feature window.\n\nTime window: a mid-panel month, month=2026-03, for feature-building — never the _sample table, which is the final month (June 2026) and would leak the future outcome window into label logic.'

In [4]:
import duckdb
from google.colab import userdata

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

hf_token = userdata.get('HF_TOKEN')
con.execute(f"""
    CREATE SECRET hf_token (
        TYPE huggingface,
        TOKEN '{hf_token}'
    );
""")

FACT = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet"
print("Connected.")

Connected.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
"""FEATURES (each knowable at the decision moment because...):
1. impressions_90d — a sum of past search impressions, all dated before the decision point.
2. clicks_90d — purely historical click counts within the feature window.
3. avg_position_90d — an average of past ranking positions; no future data involved.
4. ctr_90d — derived entirely from clicks_90d and impressions_90d, both already-observed.
5. days_since_last_seen — computed from the max report_date already in the window, relative to the window's end.

LABEL (proxy): whether a page's impressions fall over the following 30-day window — a real future outcome, unlike the starter's same-window trend_direction bucket.

CONTEXT: client_hash_id (for grouped/client-holdout validation), dim_clients.gsc_data_start / ga4_data_start (so I don't mistake "tracking hadn't started" for "no traffic").

EXCLUDED: product decision fields (health_score, priority_score, action_type, refresh flags) — not shipped anyway, named explicitly per the "Do Not Do This" rule. Also excluded: raw query/URL/title fields, and the freshest 3 days of any month (already cut from the release)."""


'FEATURES (each knowable at the decision moment because...):\n1. impressions_90d — a sum of past search impressions, all dated before the decision point.\n2. clicks_90d — purely historical click counts within the feature window.\n3. avg_position_90d — an average of past ranking positions; no future data involved.\n4. ctr_90d — derived entirely from clicks_90d and impressions_90d, both already-observed.\n5. days_since_last_seen — computed from the max report_date already in the window, relative to the window\'s end.\n\nLABEL (proxy): whether a page\'s impressions fall over the following 30-day window — a real future outcome, unlike the starter\'s same-window trend_direction bucket.\n\nCONTEXT: client_hash_id (for grouped/client-holdout validation), dim_clients.gsc_data_start / ga4_data_start (so I don\'t mistake "tracking hadn\'t started" for "no traffic").\n\nEXCLUDED: product decision fields (health_score, priority_score, action_type, refresh flags) — not shipped anyway, named explici

In [7]:
schema = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{FACT}') LIMIT 1").df()
print(schema.to_string())

                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users      BIGINT  YES  None    None  None
14      ga

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
"""Below: slice shape, availability (IS TRUE), and the deliberate leakage trap — adding one label-derived column to watch the score jump toward perfect, then removing it and keeping the honest number."""


'Below: slice shape, availability (IS TRUE), and the deliberate leakage trap — adding one label-derived column to watch the score jump toward perfect, then removing it and keeping the honest number.'

In [10]:
schema = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{FACT}') LIMIT 1").df()
print(schema.to_string())

                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users      BIGINT  YES  None    None  None
14      ga

In [12]:
schema = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{FACT}') LIMIT 1").df()
print(schema.to_string())

                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users      BIGINT  YES  None    None  None
14      ga

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
"""What this data can never tell you: month=2026-03 is an unbalanced panel — clients have different tracking start dates, and only 9 of 70 clients have 12+ months of history in the full warehouse. A single mid-panel month can't capture seasonality. Some clients may show up with partial history simply because their GSC/GA4 tracking hadn't started yet — checked against dim_clients.gsc_data_start / ga4_data_start to avoid treating "not tracked yet" as "no traffic." My feature window (all of March) and my label's target window (the following 30 days) must never overlap — this is the same discipline the leakage trap above just demonstrated."""


'What this data can never tell you: month=2026-03 is an unbalanced panel — clients have different tracking start dates, and only 9 of 70 clients have 12+ months of history in the full warehouse. A single mid-panel month can\'t capture seasonality. Some clients may show up with partial history simply because their GSC/GA4 tracking hadn\'t started yet — checked against dim_clients.gsc_data_start / ga4_data_start to avoid treating "not tracked yet" as "no traffic." My feature window (all of March) and my label\'s target window (the following 30 days) must never overlap — this is the same discipline the leakage trap above just demonstrated.'

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.